# Pears 6

### Importing things 

In [6]:
import statsmodels.api as sm
import numpy as np
import warnings
import sys 
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

script_path = os.path.abspath(os.path.join('..', 'scripts'))
if script_path not in sys.path:
    sys.path.append(script_path)

from spread import SPREAD

warnings.filterwarnings("ignore")

### Reading in data

In [7]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months]
]

# Bump volume since we have 3x the data
builder = SPREAD(threshold=1000) 
df = builder.build(my_files)

print(df.head(5))

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/audusd_dukascopy_ask_202504.parquet'

### Running it 

In [ ]:
print("--- 1. ENGLE-GRANGER COINTEGRATION ---")
# We MUST use levels (Log_A and Log_B) to find the cointegration ratio
Y = df['Log_A']
X = sm.add_constant(df['Log_B'])
ols_model = sm.OLS(Y, X).fit()

beta = ols_model.params['Log_B']
alpha = ols_model.params['const']
print(f"Hedge Ratio (Beta): {beta:.4f}")

# Calculate the actual Spread levels (We need this for the Z-Score trading signal later)
spread = Y - (beta * df['Log_B'] + alpha)

print("\n--- 2. VOLATILITY DIFFERENCING & SCALING ---")
# Because you upgraded the SPREAD class, we can calculate the spread differences 
# directly using your new Return columns!
spread_diff = df['Return_A'] - (beta * df['Return_B'])

# Scale to Basis Points
spread_diff_scaled = spread_diff * 10000 

# Add Jitter (Anti-Zero-Variance trick)
np.random.seed(42)
jitter = np.random.normal(0, 1e-4, size=len(spread_diff_scaled))
spread_diff_ready = spread_diff_scaled + jitter

print(f"Data prepared: {len(spread_diff_ready)} rows ready for optimizer.")

print("\n--- 3. MARKOV REGIME FITTING ---")
ms_model = sm.tsa.MarkovRegression(
    spread_diff_ready, 
    k_regimes=2, 
    switching_variance=True, 
    trend='c'
).fit(disp=False) 

print("Success: Markov Model fitted perfectly!")

--- 1. ENGLE-GRANGER COINTEGRATION ---
Hedge Ratio (Beta): -0.3992

--- 2. VOLATILITY DIFFERENCING & SCALING ---
Data prepared: 4255 rows ready for optimizer.

--- 3. MARKOV REGIME FITTING ---
Success: Markov Model fitted perfectly!
